# 3. langgraph로 라우터 포함된 그래프를 구성해보세요.
- reports.json, work_logs_large.csv를 사용하세요.
- csv는 duckdb에 넣어보세요.
- 그래프를 시각화하세요
- 라우터를 사용하세요.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!pkill -9 ollama

import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 61399 )


In [ ]:
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text

In [ ]:
!pip install -q --upgrade \
  langchain langchain-core langchain-community \
  langchain_chroma langchain-ollama \
  chromadb sentence-transformers \
  opentelemetry-api==1.27.0 opentelemetry-sdk==1.27.0

!pip install -q langgraph langchain-text-splitters pypdf duckdb

In [ ]:
import json, pickle, duckdb, shutil
import pandas as pd

from typing import TypedDict, Literal
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document          
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END

In [ ]:
from typing import TypedDict, Literal
import sqlite3
import pandas as pd

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama


llm = ChatOllama(    model="qwen2.5:3b",    temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

- Chroma DB 구성

In [ ]:
!mkdir -p chroma_db1

In [ ]:
# 벡터스토어 생성
vectorstore = Chroma(
    collection_name="factory_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_db1"
)

In [ ]:
# reports.json 로드
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

documents = [
    Document(
        page_content=r["report_text"],
        metadata={"report_id": r["report_id"], "source": "json"}
    )
    for r in reports
]
vectorstore.add_documents(documents)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"ChromaDB 구성 완료: {vectorstore._collection.count()}개")

ChromaDB 구성 완료: 300개


- duckdb 구성

In [ ]:
df_logs = pd.read_csv("work_logs_large.csv", encoding="utf-8-sig")

con = duckdb.connect("factory.db")
con.execute("DROP TABLE IF EXISTS work_logs")
con.execute("CREATE TABLE work_logs AS SELECT * FROM df_logs")
print(f"DuckDB 구성 완료: {con.execute('SELECT COUNT(*) FROM work_logs').fetchone()[0]}건")
con.close()

DuckDB 구성 완료: 30건


- ML모형 학습

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

con = duckdb.connect("factory.db", read_only=True)
df = con.execute("SELECT * FROM work_logs").df()
con.close()

le_설비 = LabelEncoder()
le_작업 = LabelEncoder()
df["설비명_enc"] = le_설비.fit_transform(df["설비명"])
df["작업유형_enc"] = le_작업.fit_transform(df["작업유형"])
df["불량"] = (df["작업결과"] != "완료").astype(int)

X = df[["설비명_enc", "작업유형_enc", "소요시간_h", "비용_원"]]
y = df["불량"]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

with open("defect_model.pkl", "wb") as f:
    pickle.dump({"model": model, "le_설비": le_설비, "le_작업": le_작업}, f)

print(f"ML 모델 학습 완료 | 불량 비율: {y.mean():.1%}")

ML 모델 학습 완료 | 불량 비율: 30.0%


- State 정의

In [ ]:
class AgentState(TypedDict):
    question: str
    route: str
    context: str
    answer: str

- 노드함수 정의

In [ ]:
# ── RAG 노드 ──────────────────────────────
def rag_node(state: AgentState):
    docs = retriever.invoke(state["question"])
    context = "\n".join([d.page_content for d in docs])
    return {"context": context}


# ── SQL 노드 ──────────────────────────────
def sql_node(state: AgentState):
    try:
        con = duckdb.connect("factory.db", read_only=True)
        # LLM에게 SQL 생성 요청
        sql_prompt = f"""
        아래 테이블에서 질문에 답하는 SQL을 작성하세요.
        테이블: work_logs
        컬럼: log_id, 작업일시, 설비명, 작업유형, 작업자, 소요시간_h, 작업결과, 비용_원

        질문: {state["question"]}

        SELECT 문만 출력하세요. 설명 없이.
        """
        sql = llm.invoke(sql_prompt).content.strip()
        # 코드블록 제거
        sql = sql.replace("```sql", "").replace("```", "").strip()

        result = con.execute(sql).df()
        con.close()
        return {"context": result.to_string(index=False)}
    except Exception as e:
        return {"context": f"SQL 오류: {str(e)}"}


# ── ML 예측 노드 ───────────────────────────
def ml_node(state: AgentState):
    try:
        with open("defect_model.pkl", "rb") as f:
            artifacts = pickle.load(f)
        clf = artifacts["model"]
        le_설비 = artifacts["le_설비"]
        le_작업 = artifacts["le_작업"]

        con = duckdb.connect("factory.db", read_only=True)
        df_recent = con.execute("""
            SELECT 설비명, 작업유형, 소요시간_h, 비용_원
            FROM work_logs
            ORDER BY 작업일시 DESC
            LIMIT 3
        """).df()
        con.close()

        results = []
        for _, row in df_recent.iterrows():
            설비_enc = le_설비.transform([row["설비명"]])[0]
            작업_enc = le_작업.transform([row["작업유형"]])[0]
            X = [[설비_enc, 작업_enc, row["소요시간_h"], row["비용_원"]]]
            prob = clf.predict_proba(X)[0][1]
            results.append(f"{row['설비명']} / {row['작업유형']}: 불량확률 {prob:.1%}")

        return {"context": "\n".join(results)}
    except Exception as e:
        return {"context": f"예측 오류: {str(e)}"}



In [ ]:
# ── 분류 노드 ──────────────────────────────
def classify_intent(state: AgentState):
    prompt = f"""
    다음 질문을 세 가지 중 하나로만 분류하세요.

    - rag: 불량 원인, 조치방법, 보고서 내용 검색
    - sql: 설비별 작업이력, 건수 집계, DB 조회
    - ml: 불량 예측, 위험도 분석

    질문: {state["question"]}

    rag, sql, ml 중 하나만 출력하세요.
    """
    response = llm.invoke(prompt).content.strip().lower()

    if "sql" in response:
        route = "sql"
    elif "ml" in response:
        route = "ml"
    else:
        route = "rag"

    print(f"라우팅: {route}")
    return {"route": route}


# ── 최종 답변 노드 ─────────────────────────
def answer_node(state: AgentState):
    prompt = f"""
    사용자의 질문에 아래 정보를 참고하여 한국어로 답하세요.

    질문: {state["question"]}
    참고 정보: {state["context"]}

    간결하고 명확하게 답변하세요.
    """
    response = llm.invoke(prompt)
    return {"answer": response.content}


# ── 라우터 ─────────────────────────────────
def router(state: AgentState) -> Literal["rag", "sql", "ml"]:
    return state["route"]

- 그래프 구성

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("classify", classify_intent)
builder.add_node("rag",      rag_node)
builder.add_node("sql",      sql_node)
builder.add_node("ml",       ml_node)
builder.add_node("answer",   answer_node)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    router,
    {"rag": "rag", "sql": "sql", "ml": "ml"}
)
builder.add_edge("rag",    "answer")
builder.add_edge("sql",    "answer")
builder.add_edge("ml",     "answer")
builder.add_edge("answer", END)

graph = builder.compile()
print("그래프 컴파일 완료")

그래프 컴파일 완료


In [ ]:
questions = [
    "CNC에서 Burr가 발생하는 원인은?",          # → rag
    #"프레스 P-2 작업이력 건수 알려줘",            # → sql
    #"최근 작업 데이터로 불량 위험도 예측해줘",     # → ml
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"질문: {q}")
    result = graph.invoke({"question": q})
    print(f"답변: {result['answer']}")


질문: CNC에서 Burr가 발생하는 원인은?
라우팅: rag
답변: CNC 가공에서 Burr(바우어)가 발생한 원인은 작업자가 표준 체결 순서를 따르지 않고 한쪽부터 완전 체결한 것이라고 판단됩니다. 이는 조립 순서와 중간 토크 적용 절차에 대한 교육이 필요하다는 것을 의미합니다。
